In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.incertidumbres import (
    graficar_imagen,
    crear_dataframe_incertidumbre,
    plot_incertidumbres,
    plot_combinado
)

from src.image_preprocessor import (
    ImagePreprocessor
)

from src.model_trainer import ModelTrainer
from src.image_preprocessor import ImagePreprocessor
import pickle
import pandas as pd
import tensorflow as tf

/Users/camcortes/Documents/birds-sounds/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "EfficientNetV2L" #"MobileNetV3Large" #"EfficientNetV2L" #"ResNet152V2" #"EfficientNetB7"

model_trainer = ModelTrainer(
    model_name=model_name,
    img_shape=(128, 256, 1),
    n_classes=667,
    dropout_rate=0.3,
    label_smoothing=0.1,
    fine_tune_layers=200
)

model = model_trainer.create_model()
model.summary()
model.load_weights(f"./models/weights_{model_name}.weights.h5")
tf.keras.backend.clear_session()

Model: "EfficientNetV2L_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 256, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 128, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ preprocess_input (Lambda)       │ (None, 128, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-l (Functional)   │ (None, 1280)           │   117,746,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ top_dropout (Dropout)           │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, 667)            │       854,427 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,601,275 (452.43 MB)

 Trainable params: 55,177,211 (210.48 MB)

 Non-trainable params: 63,424,064 (241.94 MB)

/Users/camcortes/Documents/birds-sounds/.venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 354 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [3]:
with open(f'./models/label_encoder_{model_name}.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

In [6]:
preprocessor = ImagePreprocessor(label_encoder=label_encoder)
data = preprocessor.load_data_from_directory("/Users/camcortes/Documents/birds-sounds/src/data/images_test/images_spectograms")
data.head()

,label,image_path
0,Acropternis orthonyx,/Users/camcortes/Documents/birds-sounds/src/data/images_test/images_spectograms/Acropternis orthonyx/121142_13.jpeg
1,Acropternis orthonyx,/Users/camcortes/Documents/birds-sounds/src/data/images_test/images_spectograms/Acropternis orthonyx/511703_5.jpeg
2,Acropternis orthonyx,/Users/camcortes/Documents/birds-sounds/src/data/images_test/images_spectograms/Acropternis orthonyx/461011_0.jpeg
3,Acropternis orthonyx,/Users/camcortes/Documents/birds-sounds/src/data/images_test/images_spectograms/Acropternis orthonyx/621780_9.jpeg
4,Acropternis orthonyx,/Users/camcortes/Documents/birds-sounds/src/data/images_test/images_spectograms/Acropternis orthonyx/428484_0.jpeg


In [7]:
import random
from tqdm import tqdm

def especie_batch(data):
    species = list(set(data["label"]))
    num_species = len(species)
    for i, specie in enumerate(tqdm(species, desc=f"Especies procesadas")):
        print(f"\nEspecie {i+1} de {(num_species)}, ({specie})")
        yield specie

def specie_ds(data, gen):
    try:
        specie = next(gen)
        df = data[data["label"] == specie]
        if df.empty:
            print(f"No more species")
            return None, None
    except StopIteration:
        return None, None
    return specie, preprocessor.create_validation_dataset(df)

In [8]:
import matplotlib.pyplot as plt
import numpy as np

def calcular_incertidumbre(dataset_espectrogramas, modelo):
    predicciones_mc = []
    confianzas_mc = []

    etiqueta_real_lista =[]
    etiqueta_inferencia_lista = []

    if dataset_espectrogramas:
        for imagenes_batch, etiquetas_batch in dataset_espectrogramas.take(1):
            num_imagenes = len(imagenes_batch)

            # MONTE CARLO DROPOUT: Múltiples pasadas con training=True
            for idx in range(num_imagenes):
                imagen = imagenes_batch[idx]
                # Añadir una dimensión de batch (necesario para la inferencia)
                imagen_batch = tf.expand_dims(imagen, axis=0)
                etiqueta_real = tf.argmax(etiquetas_batch[idx])

                etiqueta_real_lista.append(etiqueta_real.numpy())

                #for iteracion in range(5):
                logits = modelo(imagen_batch)
                probabilidades = tf.nn.softmax(logits, axis=-1).numpy()
                clase_predicha = np.argmax(probabilidades, axis=-1)[0]
                confianza = probabilidades[0, clase_predicha]

                predicciones_mc.append(clase_predicha)
                confianzas_mc.append(confianza)

                # Predicción convencional (sin dropout activo)
                logits_inferencia = modelo(imagen_batch, training=False)
                probabilidades_inferencia = tf.nn.softmax(logits_inferencia, axis=-1).numpy()
                clase_predicha_inferencia = np.argmax(probabilidades_inferencia, axis=-1)[0]
                confianza_inferencia = probabilidades_inferencia[0, clase_predicha_inferencia]

                etiqueta_inferencia_lista.append(clase_predicha_inferencia)

                #print(f"\nClase real: {etiqueta_real}")
                #print(f"Clase predicha (modo inferencia): {clase_predicha_inferencia}")
                #print(f"Confianza (modo inferencia): {confianza_inferencia:.4f} ({confianza_inferencia*100:.2f}%)")

            # Si se desea mostrar la imagen:
            # plt.imshow(imagen.numpy().squeeze(), cmap='inferno')
            # plt.title(f"Predicha: {clase_predicha_inferencia} (Inferencia)")
            # plt.axis('off')
            # plt.show()

    return predicciones_mc, confianzas_mc, etiqueta_inferencia_lista, etiqueta_real_lista

In [11]:
dataset_generado = especie_batch(data)

resultados = {}
for i in range(data["label"].nunique()):
    especie, dataset_especie = specie_ds(data, dataset_generado)
    predicciones_mc, confianzas_mc, clase_predicha_inferencia, etiqueta_real = calcular_incertidumbre(dataset_especie, model)
    resultados[especie] = {
        "clase_real": etiqueta_real,
        "clase_predicha_inferencia": clase_predicha_inferencia,
        "predicciones_mc": predicciones_mc,
        "confianzas_mc": confianzas_mc,
    }

Especies procesadas:  97%|█████████▋| 644/667 [14:36:55<13:48, 36.03s/it]


Especie 645 de 667, (Psarocolius decumanus)


Especies procesadas:  97%|█████████▋| 645/667 [14:37:44<14:33, 39.73s/it]


Especie 646 de 667, (Ara ararauna)


Especies procesadas:  97%|█████████▋| 646/667 [14:38:29<14:33, 41.58s/it]

...


Especie 665 de 667, (Hylophylax naevioides)


Especies procesadas: 100%|█████████▉| 665/667 [14:48:43<01:08, 34.41s/it]


Especie 666 de 667, (Tangara velia)


Especies procesadas: 100%|█████████▉| 666/667 [14:48:57<00:28, 28.22s/it]


Especie 667 de 667, (Grallaricula lineifrons)


In [12]:
labels = list(set(data["label"]))
labels.sort()

In [14]:
df = pd.DataFrame(resultados)
print(df.shape)
df.iloc[::1].head(2)

(4, 667)


Conirostrum bicolor  \
clase_real                 [116, 116, 116, 116, 116, 116, 116, 116, 116, 116, 116, 116, 116, 116, 116, 116, 116]   
clase_predicha_inferencia   [116, 116, 116, 116, 116, 154, 116, 116, 116, 116, 17, 116, 116, 116, 116, 116, 116]   

                                                                                                                                                                                                                                                                                                                                                                                               Legatus leucophaius  \
clase_real                 [301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301]   
clase_predicha_inferencia  [301, 301, 301, 301, 301, 301, 301, 322, 271, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 322, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 221, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 301, 376, 301, 301]   

                                                                                                                                                                                                                                                                                                                                                                                                Empidonax traillii  \
clase_real                 [195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195]   
clase_predicha_inferencia   [195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 351, 195, 195, 195, 195, 195, 195, 195, 539, 195, 195, 195, 195, 194, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 69, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195, 195]   

                                                                                                                                                                                                                                                                                                                                                                                             Pheugopedius sclateri  \
clase_real                 [426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426]   
clase_predicha_inferencia   [426, 611, 426, 426, 426, 426, 426, 425, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 260, 425, 426, 426, 316, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 426, 215, 426, 426, 426, 426, 426, 426, 426, 548, 426, 426, 426, 426, 10, 426, 426, 426, 426, 426, 426, 426, 348, 426, 426, 426, 426]   

                                                                                       

In [15]:
df = df.transpose()
df.sample(3)

,clase_real,clase_predicha_inferencia,predicciones_mc,confianzas_mc
Chlorospingus flavopectus,"[99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99]","[308, 104, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 18, 99, 100, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 10, 99, 99, 99, 99, 533, 99, 99, 99, 99, 99, 99, 35, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99]","[308, 28, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 18, 99, 100, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 10, 99, 99, 99, 99, 657, 99, 99, 99, 99, 99, 99, 35, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99]","[0.28253484, 0.30962414, 0.96790993, 0.9840513, 0.9540374, 0.9837335, 0.9934242, 0.9747457, 0.89134765, 0.983509, 0.9458998, 0.97345, 0.9762916, 0.9198865, 0.95463836, 0.7568146, 0.16095257, 0.8867246, 0.23766392, 0.98055696, 0.99542844, 0.95418864, 0.9932602, 0.97173697, 0.847373, 0.78974676, 0.93600583, 0.64909416, 0.60684663, 0.9820899, 0.9703948, 0.86731166, 0.2721231, 0.98612136, 0.90938115, 0.9620213, 0.8160574, 0.1483467, 0.975571, 0.91978425, 0.74158096, 0.9834773, 0.99753845, 0.96591705, 0.6391067, 0.92455035, 0.98513377, 0.44190785, 0.9971762, 0.97686464, 0.92952055, 0.9732519, 0.9532155, 0.9714585, 0.78453225, 0.97998327, 0.4518299, 0.9749833, 0.9022233, 0.9524117, 0.9429568, 0.9656315, 0.9870471, 0.9087125, 0.9564388, 0.9779824, 0.9994723, 0.99779654, 0.85001916, 0.9419082, 0.916338, 0.7744579, 0.95700914, 0.91563374, 0.9954591]"
Phaenostictus mcleannani,"[418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418]","[418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418]","[418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 418, 237, 418, 418, 418, 418, 418, 418, 418]","[0.9789883, 0.99753076, 0.98116714, 0.97698087, 0.963138, 0.9997135, 0.9976076, 0.41768336, 0.91581607, 0.9371754, 0.98054147, 0.9869473, 0.97952694, 0.986225, 0.9283937, 0.92534786, 0.9942842, 0.7430329, 0.98633426, 0.95645195, 0.9609241, 0.9840883, 0.98140633, 0.9823683, 0.15373181, 0.96592385, 0.9745247, 0.9429615, 0.965474, 0.9622338, 0.9858941, 0.9618403]"
Cantorchilus leucopogon,"[64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64]","[64, 64, 64, 336, 64, 64, 64, 64, 64, 64, 299, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 451, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64]","[64, 64, 64, 336, 64, 64, 64, 64, 64, 64, 299, 260, 64, 64, 64, 64, 64, 64, 64, 64, 64, 500, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64]","[0.9400944, 0.81525445, 0.8653094, 0.39862508, 0.9961224, 0.9921829, 0.65019816, 0.9995733, 0.99259335, 0.16925041, 0.57195127, 0.38863337, 0.9985208, 0.8911292, 0.7926331, 0.95740646, 0.94080365, 0.99962115, 0.52083385, 0.99121463, 0.99796605, 0.24087706, 0.98446953, 0.9744984, 0.93599313, 0.95677936, 0.401013, 0.75320476, 0.9540767, 0.62821287, 0.38149303, 0.97394806, 0.9941203, 0.98815405, 0.8575385]"


In [16]:
def calcular_proporcion(row):
    real = set(row['clase_real'])
    predicha = set(row['predicciones_mc'])

    # Elementos comunes entre ambas listas
    elementos_comunes = len(real.intersection(predicha))

    # Total de elementos únicos en clase_real
    total_elementos_real = len(real)

    # Calcula la proporción (evitando división por cero)
    if total_elementos_real > 0:
        return elementos_comunes / total_elementos_real
    else:
        return 0

# Aplicar la función a cada fila para crear la nueva columna
df['proporcion_coincidencia'] = df.apply(calcular_proporcion, axis=1)

# Si quieres expresarla como porcentaje
df['porcentaje_coincidencia'] = df['proporcion_coincidencia'] * 100

In [19]:
df.sample(3)

,clase_real,clase_predicha_inferencia,predicciones_mc,confianzas_mc,proporcion_coincidencia,porcentaje_coincidencia
Attila cinnamomeus,"[38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38]","[38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 443, 38, 38, 38, 38, 38, 173, 650, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 335, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 274, 38, 38, 38, 38, 38, 38, 38, 38]","[38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 443, 38, 38, 38, 38, 38, 173, 650, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 335, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 652, 38, 38, 38, 38, 38, 38, 38, 38]","[0.95261365, 0.9305934, 0.97200835, 0.9693832, 0.93993586, 0.90508103, 0.9483218, 0.9071486, 0.91027445, 0.85074615, 0.95901996, 0.59696525, 0.9412112, 0.7957336, 0.87349707, 0.9757204, 0.9203449, 0.9162803, 0.48308507, 0.910654, 0.9821147, 0.9740363, 0.9616561, 0.89686596, 0.9734759, 0.99908924, 0.9704325, 0.9775264, 0.9139094, 0.91274434, 0.9543904, 0.93983865, 0.9489267, 0.7238349, 0.96289384, 0.98275405, 0.94360787, 0.98129076, 0.9728742, 0.5112396, 0.9331584, 0.9924886, 0.50548774, 0.097549856, 0.97387165, 0.9246421, 0.8856968, 0.78360116, 0.87468827, 0.8018969, 0.9256215, 0.8494971, 0.50004697, 0.9158426, 0.5971613, 0.8375748, 0.94509804, 0.9934257, 0.3126503, 0.91146845, 0.9105256, 0.8169531, 0.84709454, 0.28828284, 0.95786893, 0.9583147, 0.2467548, 0.96393746, 0.59641373, 0.7020411, 0.90136504, 0.984363, 0.94457436, 0.91967463, 0.9146748]",1.00,100.00
Sipia laemosticta,"[530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530]","[380, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 560, 530, 148, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530]","[409, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 560, 530, 148, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530, 530]","[0.28483278, 0.967447, 0.9626155, 0.972982, 0.97960633, 0.98107076, 0.99735093, 0.96789885, 0.87866616, 0.99380726, 0.9515299, 0.9764389, 0.99728084, 0.97678214, 0.999461, 0.96243304, 0.97351396, 0.90893316, 0.98437136, 0.96375144, 0.9971208, 0.99529797, 0.9775857, 0.96969056, 0.12700328, 0.9897078, 0.48015502, 0.9976419, 0.9876409, 0.7225641, 0.9500993, 0.9786673, 0.99254453, 0.97822565, 0.99005497, 0.9208368, 0.94439113, 0.99174786, 0.9840776, 0.98714674, 0.89838123, 0.9780828, 0.999759, 0.9827849, 0.9919991, 0.9761386, 0.9620443, 0.88128144, 0.9676283, 0.96163154, 0.98962295, 0.95145947, 0.6941033, 0.99135405]",1.00,100.00
Tyrannus savana,"[638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638]","[638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 494, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 303, 638, 638, 638, 638, 638, 638]","[638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 3, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 638, 303, 638, 638, 638, 638, 638, 638]","[0.9871738, 0.9239484, 0.92452484, 0.84486985, 0.8852404, 0.9645868, 0.96691287, 0

In [23]:
df.to_csv("/Users/camcortes/Documents/birds-sounds/src/data/incertidumbres_EfficentNetV2L.csv")